# Data augmentation strategies — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def rate(x): return float(np.mean(np.asarray(x)))
def nearest_predict(train_x, train_y, test_x):
    train_x=np.asarray(train_x); train_y=np.asarray(train_y); preds=[]
    for t in np.asarray(test_x):
        preds.append(train_y[int(np.argmin(np.abs(train_x-t)))])
    return np.asarray(preds)
print('setup ok')

## Changing examples while preserving labels
Augmentation encodes invariances: which changes should not change the label. These examples show small jitter, variance reduction, a harmful large transform, mixup, and an end-to-end augmented boundary.

In [ ]:
# 1) Small jitter preserves a tabular value's label-relevant location
x=np.array([1.,2.,3.]); aug=x+np.array([.1,-.1,.2])
plt.figure(figsize=(6,3)); plt.scatter(x,np.zeros_like(x),label='original'); plt.scatter(aug,np.ones_like(aug),label='augmented'); plt.yticks([0,1],['orig','aug']); plt.legend(); plt.title('small perturbations stay nearby')
assert abs(np.r_[x,aug].mean()-2.0333333333)<1e-9

In [ ]:
# 2) Averaging predictions over augmentations reduces noise
preds=np.array([.62,.58,.65,.55,.60]); avg=preds.mean()
plt.figure(figsize=(6,3)); plt.plot(preds,'o-'); plt.axhline(avg,c='r'); plt.title(f'augmentation average = {avg:.2f}')
assert abs(avg-0.6)<1e-12

In [ ]:
# 3) Too much augmentation crosses the label boundary
x=np.linspace(-1,1,80); y=(x>0).astype(int); shifted=x+.8; flips=((shifted>0).astype(int)!=y).mean()
plt.figure(figsize=(6,3)); plt.plot(x,y,label='original label'); plt.plot(x,(shifted>0).astype(int),label='after big shift'); plt.legend(); plt.title('large shift no longer preserves labels')
assert abs(flips-0.4)<1e-12

In [ ]:
# 4) Mixup creates convex combinations of inputs and labels
x1,x2=np.array([0.,0.]),np.array([2.,2.]); lam=.3; xm=lam*x1+(1-lam)*x2; ym=lam*0+(1-lam)*1
plt.figure(figsize=(4,4)); plt.plot([x1[0],x2[0]],[x1[1],x2[1]],'k--'); plt.scatter([x1[0],x2[0],xm[0]],[x1[1],x2[1],xm[1]],s=[60,60,90]); plt.title(f'mixup label = {ym:.1f}')
assert np.allclose(xm,[1.4,1.4]) and abs(ym-0.7)<1e-12

In [ ]:
# 5) Augmentation fills gaps around a sparse class cluster
rng=np.random.default_rng(4); base=rng.normal([1,1],.12,(5,2)); aug=np.vstack([base+rng.normal(0,.08,base.shape) for _ in range(8)])
plt.figure(figsize=(4,4)); plt.scatter(base[:,0],base[:,1],label='original'); plt.scatter(aug[:,0],aug[:,1],alpha=.35,label='augmented'); plt.legend(); plt.title('local cloud around sparse examples')
assert aug.shape==(40,2)